# 11.13 多個字怎麼依序讀出？

# 第 11 章：對齊、問答與語言能力保留

前置：第10章介面與第7章SFT。能力實驗需先有「文字屬性→正確答案」基模，以及可分類的視覺encoder；用scripts/prepare_data.py和pretrain_encoders.py準備。下面隨機模型只驗證資料／梯度，所有品質留給讀者訓練後量測。




In [ ]:
from pathlib import Path
import sys

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("先依 course/README.md 下載並開啟這個 repo")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：圖片切小方塊：先檢查能不能拼回來**

16×16 RGB，patch邊長4，得到16個patch。

橘色邊框只提示閱讀順序，靜態標註一直保留。系統的減少動態效果設定會停用動畫。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/patchify.svg")))

**先想一想：**每個字90%正確，5位整串會有90%正確嗎？

多個數字串接後，目標變成有順序的字符序列。每字符準確率和整串準確率要分開；圖像裁切可能直接讓答案缺字。

**把直覺寫成數學：**exact_match要求所有字符一致；CER爲編輯距離/標準答案長度。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
truth = "1010"
predicted = "1000"
char_accuracy = sum(a == b for a, b in zip(truth, predicted)) / len(truth)
print("字符準確率", char_accuracy, "整串正確", truth == predicted)
print("獨立錯誤假設下，5位全對約", 0.9**5)

**如何讀結果：**獨立假設只是直覺；真實OCR錯誤可能相關，需直接算完整串。

**只改一件事：**只增加字串長度，記錄整串指標變化。
